<a href="https://colab.research.google.com/github/digitalbonebone/tools/blob/main/Obiwa%E7%9A%84%E7%A8%8B%E5%BA%8F%E5%9D%97.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# 解析花名册excel 制作技能考生tmp_stu数据
# 1) 上传 Excel 文件
# =========================
from google.colab import files
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

# =========================
# 2) 安装并导入依赖
# =========================
# Colab 一般已带 openpyxl，这里保守写法
!pip -q install pandas openpyxl

import pandas as pd
from collections import defaultdict

# =========================
# 3) 读取第一个 Sheet
# =========================
df = pd.read_excel(file_name, sheet_name=0)

# 清理列名首尾空格
df.columns = df.columns.astype(str).str.strip()

# 检查必要列
required_cols = ['学号', '姓名']
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Excel 中缺少列: {missing_cols}，当前列名为: {list(df.columns)}")

# 只保留需要的列
df = df[['学号', '姓名']].copy()

# 去掉空行
df = df.dropna(subset=['学号', '姓名'])

# 学号转字符串，避免被识别成数字/小数
df['学号'] = df['学号'].astype(str).str.strip()
df['姓名'] = df['姓名'].astype(str).str.strip()

# 如果学号像 3220105389.0 这种，去掉末尾 .0
df['学号'] = df['学号'].str.replace(r'\.0$', '', regex=True)

# 去掉学号或姓名为空字符串的行
df = df[(df['学号'] != '') & (df['姓名'] != '')].reset_index(drop=True)

# =========================
# 4) 工具函数
# =========================
def sql_escape(text: str) -> str:
    """转义 SQL 单引号"""
    return str(text).replace("'", "''")

def duplicate_suffix(n: int) -> str:
    """
    第2次重复 -> B
    第3次重复 -> C
    ...
    若重复很多，超过 Z 后继续 AA/AB... 这种风格
    这里传入的是“第几次出现”，例如：
      1 -> ''   (首次，不加后缀)
      2 -> 'B'
      3 -> 'C'
      ...
    """
    if n <= 1:
        return ''

    # 把 2 -> B, 3 -> C ... 26 -> Z, 27 -> AA, 28 -> AB ...
    x = n - 2  # 从 B 开始映射
    letters = []
    while True:
        letters.append(chr(ord('A') + (x % 26)))
        x = x // 26 - 1
        if x < 0:
            break
    return ''.join(reversed(letters))

def build_insert_sql(rows, table_name='tmp_stu', columns=('stucode', 'name'), batch_size=1000):
    """
    将 [(stucode, name), ...] 构造成多段 INSERT SQL
    """
    if not rows:
        return "-- 无数据\n"

    sql_blocks = []
    col_list = ", ".join(columns)

    for i in range(0, len(rows), batch_size):
        chunk = rows[i:i+batch_size]
        values_sql = ",\n".join(
            f"('{sql_escape(stucode)}','{sql_escape(name)}')"
            for stucode, name in chunk
        )

        block = (
            f"INSERT INTO {table_name} ({col_list})\n"
            f"SELECT * FROM (VALUES\n"
            f"{values_sql}\n"
            f") AS t({col_list});"
        )
        sql_blocks.append(block)

    return "\n\n".join(sql_blocks)

# =========================
# 5) 区分主数据和重复数据
# =========================
seen_count = defaultdict(int)

main_rows = []       # 首次出现的学号
duplicate_rows = []  # 重复学号（加后缀后）

for _, row in df.iterrows():
    stu_code = row['学号']
    stu_name = row['姓名']

    seen_count[stu_code] += 1
    appear_no = seen_count[stu_code]

    if appear_no == 1:
        # 第一次出现，放主语句
        main_rows.append((stu_code, stu_name))
    else:
        # 重复出现，尾部加 B/C/D...
        new_code = stu_code + duplicate_suffix(appear_no)
        duplicate_rows.append((new_code, stu_name))

# =========================
# 6) 生成 SQL 文本
# =========================
sql_parts = []

sql_parts.append("-- =============================================")
sql_parts.append("-- 第一段：主数据（每个学号首次出现）")
sql_parts.append("-- =============================================")
sql_parts.append(build_insert_sql(main_rows, table_name='tmp_stu', columns=('stucode', 'name')))

sql_parts.append("")
sql_parts.append("-- =============================================")
sql_parts.append("-- 第二段：重复学号数据")
sql_parts.append("-- 说明：当学号重复时，从第2次出现开始，在学号尾部追加字母 B/C/D/... ")
sql_parts.append("-- 示例：")
sql_parts.append("--   原学号 3220105389 第1次出现 -> 3220105389")
sql_parts.append("--   原学号 3220105389 第2次出现 -> 3220105389B")
sql_parts.append("--   原学号 3220105389 第3次出现 -> 3220105389C")
sql_parts.append("-- =============================================")
sql_parts.append(build_insert_sql(duplicate_rows, table_name='tmp_stu', columns=('stucode', 'name')))

final_sql = "\n".join(sql_parts)

# =========================
# 7) 保存 SQL 文件
# =========================
output_file = "tmp_stu_insert.sql"
with open(output_file, "w", encoding="utf-8") as f:
    f.write(final_sql)

print(f"SQL 文件已生成: {output_file}")
print(f"主数据条数: {len(main_rows)}")
print(f"重复数据条数: {len(duplicate_rows)}")

# 如需预览前1000字符
print("\n===== SQL预览（前1000字符）=====\n")
print(final_sql[:1000])

# =========================
# 8) 下载文件
# =========================
files.download(output_file)

Saving 1.xlsx to 1.xlsx
SQL 文件已生成: tmp_stu_insert.sql
主数据条数: 187
重复数据条数: 0

===== SQL预览（前1000字符）=====

-- =============================================
-- 第一段：主数据（每个学号首次出现）
-- =============================================
INSERT INTO tmp_stu (stucode, name)
SELECT * FROM (VALUES
('3220105389','赵梦茹'),
('3220105067','张雅思'),
('3220105081','李静文'),
('3220105392','袁佳'),
('3220102133','王沛然'),
('3210101773','许修齐'),
('3220104206','张家楷'),
('3220101989','莫丽婷'),
('3220104919','刘明远'),
('3220105818','赵家棋'),
('3220105068','陈帝丞'),
('3220104272','程焯'),
('3220105393','胡益铭'),
('3220102127','雪傲函'),
('3220101991','黄牧云'),
('3220104712','关子昂'),
('3220105129','韩佳航'),
('3220106430','白欣宜'),
('3220105130','乔楚涵'),
('3220104277','裘颖祯'),
('3220101488','杨道远'),
('3220106149','卡米拉·阿不都瓦克'),
('3220100994','徐晨曦'),
('3220104237','盛镠瑜'),
('3220105396','常清岚'),
('3220105272','王语涵'),
('3220105079','武文麒'),
('3220101820','张晶晶'),
('3220106080','洪昀征'),
('3220106072','王东瀚'),
('3220105135','王亦心'),
('3220104316','李旻洋'),
('3210104814

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# -*- coding: utf-8 -*-
# 通用 JSON 转 Excel（Google Colab 可直接运行）

import json
import os
import pandas as pd
from google.colab import files

# =========================
# 配置区
# =========================

# 方式1：是否使用“直接粘贴 JSON”
USE_PASTED_JSON = False

# 如果 USE_PASTED_JSON = True，把 JSON 粘贴到下面
JSON_TEXT = r'''
[
  {"id": "001", "name": "张三", "age": 18},
  {"id": "002", "name": "李四", "age": 20}
]
'''

# 输出文件名
OUTPUT_FILE = "json_to_excel_result.xlsx"

# 哪些字段强制按文本保存，避免 Excel 把 001 变成 1
TEXT_COLUMNS = ["ORGBatchCode", "id", "code"]

# 如果 JSON 顶层是对象，而真正的数据数组在某个字段里，可在这里指定
# 例如：{"data":[...]} 就填 "data"
ROOT_ARRAY_KEY = None

# =========================
# 工具函数
# =========================

def load_json_data():
    """读取 JSON 数据：支持粘贴字符串或上传文件"""
    if USE_PASTED_JSON:
        text = JSON_TEXT.strip()
        if not text:
            raise ValueError("JSON_TEXT 为空，请粘贴 JSON 数据。")
        return json.loads(text), "pasted_json"
    else:
        uploaded = files.upload()
        if not uploaded:
            raise ValueError("未上传任何文件。")
        filename = list(uploaded.keys())[0]
        content = uploaded[filename].decode("utf-8")
        return json.loads(content), os.path.splitext(filename)[0]

def normalize_to_dataframe(data):
    """
    将 JSON 数据规范化为 DataFrame
    支持：
    1. 顶层为列表：[{}, {}, ...]
    2. 顶层为对象：{...}
    3. 顶层对象中某个字段是列表：{"data":[...]}
    """
    if ROOT_ARRAY_KEY and isinstance(data, dict):
        data = data.get(ROOT_ARRAY_KEY, [])

    if isinstance(data, list):
        # 顶层是数组
        if len(data) == 0:
            return pd.DataFrame()
        return pd.json_normalize(data, sep=".")

    elif isinstance(data, dict):
        # 顶层是对象
        # 若对象内部存在列表字段，优先尝试展开
        list_keys = [k for k, v in data.items() if isinstance(v, list)]
        if len(list_keys) == 1:
            return pd.json_normalize(data[list_keys[0]], sep=".")
        else:
            return pd.json_normalize([data], sep=".")

    else:
        raise ValueError("不支持的 JSON 结构，顶层必须是 list 或 dict。")

def clean_dataframe(df):
    """清洗 DataFrame"""
    if df.empty:
        return df

    for col in df.columns:
        # 字符串列去空格
        if df[col].dtype == "object":
            df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

    # 指定列按文本处理
    for col in TEXT_COLUMNS:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: "" if pd.isna(x) else str(x))

    return df

def export_to_excel(df, output_file):
    """导出 Excel，并做基础格式处理"""
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="data")
        ws = writer.sheets["data"]

        # 冻结首行
        ws.freeze_panes = "A2"

        # 自动列宽
        for col_cells in ws.columns:
            max_len = 0
            for cell in col_cells:
                try:
                    value_len = len(str(cell.value)) if cell.value is not None else 0
                    if value_len > max_len:
                        max_len = value_len
                except:
                    pass
            ws.column_dimensions[col_cells[0].column_letter].width = min(max(max_len + 2, 10), 60)

        # 文本列格式设为文本
        for col_name in TEXT_COLUMNS:
            if col_name in df.columns:
                col_index = list(df.columns).index(col_name) + 1
                for row in range(2, len(df) + 2):
                    ws.cell(row=row, column=col_index).number_format = "@"

def main():
    data, base_name = load_json_data()
    df = normalize_to_dataframe(data)
    df = clean_dataframe(df)

    if df.empty:
        print("JSON 中没有可导出的数据。")
        return

    output_file = OUTPUT_FILE or f"{base_name}.xlsx"
    export_to_excel(df, output_file)

    print(f"转换完成：{output_file}")
    print(f"行数：{len(df)}")
    print(f"列数：{len(df.columns)}")
    print("字段：", list(df.columns))

    files.download(output_file)

# 执行
main()

Saving req20260 v1.0.json to req20260 v1.0.json
转换完成：json_to_excel_result.xlsx
行数：586
列数：15
字段： ['adOrgbatchcode', 'kq', 'kd', 'Name', 'orgName', 'baseName', 'Projecttype', 'ProjectCode', 'Candidate', 'Unit_1', 'Unit_2', 'Unit_3', 'Unit_4', 'Unit_5', 'Unit_6']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# excel 分页 安装必要库
!pip install openpyxl --quiet

import pandas as pd
from google.colab import files

# 上传文件
uploaded = files.upload()

# 假设只上传了一个文件，取第一个文件名
filename = list(uploaded.keys())[0]
print(f"已上传文件: {filename}")

# 读取 Excel 的 Sheet1
# df = pd.read_excel(filename, sheet_name='Sheet1') # Original line

xls = pd.ExcelFile(filename)
sheet_names = xls.sheet_names
print(f"Excel 文件 '{filename}' 中包含的 Sheet 名称: {sheet_names}")

if 'Sheet1' in sheet_names:
    df = pd.read_excel(filename, sheet_name='Sheet1')
    print("已成功读取 'Sheet1'.")
elif sheet_names:
    first_sheet = sheet_names[0]
    print(f"未找到 'Sheet1', 默认读取第一个 Sheet: '{first_sheet}'.")
    df = pd.read_excel(filename, sheet_name=first_sheet)
else:
    raise ValueError(f"Excel 文件 '{filename}' 不包含任何 Sheet。")

# 显示前几行确认
print("原始数据预览:")
print(df.head())

# 依据A列分组，A列假设列名为第1列名，或者就是 'A' 或第0列
# 先获取A列列名（假设是第1列）
colA = df.columns[0]
print(f"A列名称: {colA}")

# 新建一个Excel writer对象
with pd.ExcelWriter('分组结果.xlsx', engine='openpyxl') as writer:
    # 按A列分组
    groups = df.groupby(colA)
    for group_name, group_data in groups:
        # group_name 是分组依据的值（比如数值）
        # group_data 是分组后的数据
        sheet_name = str(group_name)
        # sheet名称不能超过31字符，且不能含有某些特殊字符
        # 简单处理一下：
        sheet_name = sheet_name[:31].replace(':','').replace('/','').replace('\\','').replace('?','').replace('*','').replace('[','').replace(']','')

        # 保存该组数据到对应sheet
        group_data.to_excel(writer, sheet_name=sheet_name, index=False)

print("分组Excel文件已保存为 '分组结果.xlsx'。")

# 提供下载
files.download('分组结果.xlsx')

Saving hmc202603 截至25成绩.xlsx to hmc202603 截至25成绩 (1).xlsx
已上传文件: hmc202603 截至25成绩 (1).xlsx
Excel 文件 'hmc202603 截至25成绩 (1).xlsx' 中包含的 Sheet 名称: ['Sheet1', 'Sheet2']
已成功读取 'Sheet1'.
原始数据预览:
                             PaperTitle        Code Name  Specialty  class  \
0  HMC-2025-2026-2期初《儿科学》影像医学、麻醉医学专业补考卷  1206220521  汪玮丞        NaN    NaN   
1  HMC-2025-2026-2期初《儿科学》影像医学、麻醉医学专业补考卷  1206230318  贾国曦        NaN    NaN   
2  HMC-2025-2026-2期初《儿科学》影像医学、麻醉医学专业补考卷  1206230126  戎一帆        NaN    NaN   
3  HMC-2025-2026-2期初《儿科学》影像医学、麻醉医学专业补考卷  1206230519  余俊毅        NaN    NaN   
4  HMC-2025-2026-2期初《儿科学》影像医学、麻醉医学专业补考卷  1206230709  步宇豪        NaN    NaN   

   admissions_specialty  S_area  Gscore  Grank  
0                   NaN     NaN    79.0      1  
1                   NaN     NaN    76.0      2  
2                   NaN     NaN    74.0      3  
3                   NaN     NaN    72.0      4  
4                   NaN     NaN    71.0      5  
A列名称: PaperTitle
分组Excel文件已保存为 '分组结果.xlsx'。


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>